# Operations Reasearch: Examples for Gurobipy

## Makespam Minimization Problem

Recall the following parallel machine makespan minimization problem we introduced in the past. There are m machines and n jobs. For job j, we know its processing time is $p_j$. We want to assign each job to a machine so that the makespan, which is the longest total processing time among all machines, is minimized. For example, suppose that we have n = 15 jobs and m = 3 machines. The processing times are listed in Table 1. These data define an instance for the parallel machine makespan minimization problem.

**Table 1. Data for Problem 4**
| Job | Processing time  |
|-----|------------------|
| 1   | 7                |
| 2   | 4                |
| 3   | 6                |
| 4   | 9                |
| 5   | 12               |
| 6   | 6                |
| 7   | 10               |
| 8   | 11               |
| 9   | 8                |


While this problem is NP-hard, it is straightforward to consider the following heuristic algorithm. We simply assign each job one by one to a machine. When a job is to be assigned a machine, it is assigned the machine currently with the smallest total processing time (if there is a tie, choose the machine with the smallest machine ID among those machines having the smallest total process time.

For example, suppose that we assign jobs to machines in the order the job IDs, from small to large. We will then assign job 1 to machine 1, job 2 to machine 2, job 3 to machine 3, and then job 4 to machine 2 (because at that moment machine 2 has the smallest total processing time, which is 4), and then job 5 to machine 3, ..., and finally job 15 to machine 3. The result of applying the simple heuristic algorithm is in Table 2. The makespan of the simple heuristic algorithm on this instance is 45.


**Table 2: Result of applying the simple heuristic algorithm**
| Machine | Jobs assigned to this machine | Total processing time     |
|---------|-------------------------------|---------------------------|
| 1       | 1, 6, 7, 10, 13               | 7 + 6 + 10 + 7 + 15 = 45  |
| 2       | 2, 4, 8, 11, 14               | 4 + 9 + 11 + 6 + 14 = 44  |


In [ ]:
from gurobipy import *
import pandas as pd
from pathlib import Path

In [ ]:
#get path of current notebook
CURRENT_DIR = Path.cwd()
DATA_DIR = CURRENT_DIR / "../data"

In [ ]:
# Read excel sheets and transform them into lists and matrices
basic_info = pd.read_excel(DATA_DIR / 'OR2_Week3_dataset.xlsx', 'Basic information')
cities = range(len(basic_info['City']))
markets = range(len(basic_info['Market']))

city_info = pd.read_excel(DATA_DIR / 'OR2_Week3_dataset.xlsx', 'City\'s information')
operating_costs = city_info['Operating cost']
capacities = city_info['Capacity']

market_info = pd.read_excel(DATA_DIR / 'OR2_Week3_dataset.xlsx', 'Market\'s information')
demands = market_info['Demand']

shipping_info = pd.read_excel(DATA_DIR / 'OR2_Week3_dataset.xlsx', 'Shipping cost', index_col = 0)
shipping_costs = []
for i in shipping_info.index:
    shipping_costs.append(list(shipping_info.loc[i]))

In [ ]:
eg2 = Model("eg2")    # build a new model
    
# add variables as a list
x = []
for j in cities:
    x.append(eg2.addVar(lb = 0, vtype = GRB.BINARY, name = "x" + str(j+1)))
             
y = []
for i in markets:
    y.append([])
    for j in cities:
        y[i].append(eg2.addVar(lb = 0, vtype = GRB.CONTINUOUS, name = "y" + str(i+1) + str(j+1)))
        
# setting the objective function 
eg2.setObjective(
    quicksum(operating_costs[j] * x[j] for j in cities) 
    +  quicksum(quicksum(shipping_costs[i][j] * y[i][j] for j in cities) for i in markets)
    , GRB.MINIMIZE) 

# add constraints and name them
eg2.addConstrs((quicksum(y[i][j] for i in markets) <= capacities[j] * x[j] 
                for j in cities), "productCapacity")

eg2.addConstrs((quicksum(y[i][j] for j in cities) >= demands[i]
                for i in markets), "demand_fulfillment")


    
eg2.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M5
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 11 rows, 30 columns and 60 nonzeros (Min)
Model fingerprint: 0x4a930afa
Model has 30 linear objective coefficients
Variable types: 25 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [2e+00, 4e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+00, 2e+04]

Presolve time: 0.00s
Presolved: 11 rows, 30 columns, 60 nonzeros
Variable types: 25 continuous, 5 integer (5 binary)
Found heuristic solution: objective 313600.00000

Root relaxation: objective 2.744000e+05, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 274400.000    0    4 313600.000 274400.000  12

In [ ]:
print("Result:")

for j in cities:
    print(x[j].varName, '=', x[j].x)
# head of the result table
print("\tMarket1\tMarket2\tMarket3\tMarket4\tMarket5")

for j in cities:
    # mark which product is printed now
    print("City" + str(j+1), "\t", end="")
    for i in markets:
        # print values of each kind of product
        if len(str(y[i][j].x)) < 7:
            print(y[i][j].x, "\t", end="")
        else:
            print(y[i][j].x, "", end="")
    print("")    # use for change line

print("z* =", eg2.objVal)    # print objective value

Result:
x1 = 0.0
x2 = 1.0
x3 = 1.0
x4 = 1.0
x5 = 1.0
	Market1	Market2	Market3	Market4	Market5
City1 	0.0 	0.0 	0.0 	0.0 	0.0 	
City2 	8000.0 	12000.0 0.0 	0.0 	0.0 	
City3 	0.0 	0.0 	9000.0 	0.0 	0.0 	
City4 	0.0 	0.0 	0.0 	0.0 	17000.0 
City5 	0.0 	0.0 	0.0 	14000.0 0.0 	
z* = 280400.0


***Exercise*** Suppose now the company wants to set up at least 4 centers for some reasons, how should we modify the model?

In [ ]:
# add a new constraint
eg2.addConstr(quicksum(x[j] for j in cities) >= 4, "min_4_cities")